<a href="https://colab.research.google.com/github/FaizaMahmud/u2287019_IB2D40/blob/main/u2287019_IB2D40.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns



### STEP 1: DATA FORMATTING
class AirQualityCleaner:
    """
    A class to clean, analyse and visualise air quality data from London.

    This class handles the entire data analysis pipeline from loading raw Excel data
    to generating insights and visualisations about urban pollution patterns.
    """

    def __init__(self, file_path, sheet_name, skiprows=1):
        """
        Initialise the cleaner with dataset information.

        Parameters:
            file_path (str): Path to the Excel file containing air quality data
            sheet_name (str): Name of the worksheet to analyse
            skiprows (int): Number of header rows to skip when loading data
        """
        # Load the dataset into a pandas DataFrame
        self.data = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=skiprows)

        # Define columns containing pollution measurements for consistent access
        self.pollutant_cols = [
            'Nitric Oxide (ug/m3)',
            'Nitrogen Dioxide (ug/m3)',
            'Oxides of Nitrogen (ug/m3)',
            'Ozone (ug/m3)',
            'PM10 Particulate (ug/m3)',
            'PM2.5 Particulate (ug/m3)',
            'Sulphur Dioxide (ug/m3)'
        ]

    def clean_dates(self):
        """
        Standardise date formatting and convert to datetime objects

        Transforms various date formats into a consistent datetime column.
        Handles both string and datetime input formats for maximum compatibility.
        """
        def clean_date(date_value):
            """
            Convert date values to a standardized string format.

            Handles both string dates (e.g., 'Dec-09') and datetime objects
            by extracting month abbreviation and year, then standardizing the format.

            Parameters:
                date_value: String or datetime object representing a date

            Returns:
                str: Standardised date string in 'MMM-YYYY' format
            """
            if isinstance(date_value, str):
                # Extract first three characters as month and last two as year
                return date_value[:3].upper() + '-20' + date_value[-2:]
            else:
                # Handle datetime objects by formatting appropriately
                return date_value.strftime('%b-%Y')[:3].upper() + '-20' + date_value.strftime('%b-%Y')[-2:]

        # Apply standardisation function to all date entries
        self.data['Month'] = self.data['Month (text)'].apply(clean_date)

        # Convert standardised strings to proper datetime objects for time-based analysis
        self.data['Month'] = pd.to_datetime(self.data['Month'], format='%b-%Y')

        # Remove original text column to avoid redundancy
        self.data = self.data.drop('Month (text)', axis=1)



### STEP 2: DATA CLEANING
    def clean(self):
        """
        Clean pollution data by handling missing values and ensuring non-negative measurements.

        Replaces NULL/NaN values with zero and converts negative measurements
        to positive values, since negative pollution levels are physically impossible.
        """
        for col in self.pollutant_cols:
            # Replace missing values with 0 and convert negatives to positives in one step
            self.data[col] = self.data[col].fillna(0).abs()

    def save(self, output_path):
        """
        Save the cleaned dataset to a CSV file.

        Parameters:
            output_path (str): Destination path for the cleaned CSV file
        """
        self.data.to_csv(output_path, index=False)
        print(f"Cleaned data saved to {output_path}")



### STEP 3: DATA ANALYSIS
# INSIGHT 1 - Monthly averages with peak annotation
    def monthly_pollution_trends(self, save_as=None, img_format='png'):
                """
        Analyse and visualise monthly pollution trends with peak highlighting.

        Calculates monthly averages for all pollutants and identifies the peak
        NO₂ period, highlighting it with an annotation to draw attention to
        the most critical pollution episode.

        Parameters:
            save_as (str, optional): Base filename for saving the visualization
            img_format (str, optional): Image format for saved plot ('png', 'pdf', etc.)

        Returns:
            DataFrame: Monthly average pollution levels for all pollutants
        """
                # Calculate monthly averages using end-of-month aggregation
                monthly_avg = self.data.groupby(pd.Grouper(key='Month', freq='ME'))[self.pollutant_cols].mean()

                # Create visualisation with sufficient size for detail
                plt.figure(figsize=(14, 7))
                ax = sns.lineplot(data=monthly_avg)
                ax.set_xlim(monthly_avg.index.min(), monthly_avg.index.max())

                # Identify and annotate the month with highest NO₂ concentration
                no2_peak = monthly_avg['Nitrogen Dioxide (ug/m3)'].idxmax()
                peak_value = monthly_avg.loc[no2_peak, 'Nitrogen Dioxide (ug/m3)']

                # Add explanatory annotation with arrow pointing to the peak
                ax.annotate(f'Peak NO₂: {peak_value:.1f} µg/m³\n({no2_peak.strftime("%b %Y")})',
                        xy=(no2_peak, peak_value),  # Point to annotate
                        xytext=(no2_peak + pd.DateOffset(months=2), peak_value),    # Text position
                        arrowprops=dict(facecolor='red', shrink=0.05))  # Arrow style

                # Improved x-axis formatting
                ax.xaxis.set_major_locator(mdates.YearLocator(1))  # Yearly ticks
                ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))  # Monthly minor ticks
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  # Show full year

                # Add clear labels and legend for interpretation
                plt.title('Monthly Air Pollution Trends with Seasonal Patterns')
                plt.ylabel('Concentration (µg/m³)')
                plt.xlabel('Year')
                plt.legend(title='Pollutants', labels=self.pollutant_cols, bbox_to_anchor=(1.05, 1))
                plt.grid(True, which='both', linestyle='--', linewidth=0.5)

                # Save visualisation if filename provided
                if save_as:
                    plt.savefig(f"{save_as}_monthly_trends.{img_format}", bbox_inches='tight')
                plt.show()
                return monthly_avg

# INSIGHT 2 - Hourly trends with traffic rush highlights
    def analyse_hourly_trends(self, save_as=None, img_format='png'):
        """
        Analyse pollution patterns by hour of day, highlighting rush hour impacts.

        Examines how pollutant concentrations vary throughout a typical day
        and highlights morning and evening rush hour periods to identify
        traffic-related pollution patterns.

        Parameters:
            save_as (str, optional): Base filename for saving the visualisation
            img_format (str, optional): Image format for saved plot ('png', 'pdf', etc.)

        Returns:
            DataFrame: Hourly average pollution levels for all pollutants
        """
        # Extract hour component from time strings
        self.data['Hour'] = self.data['GMT'].str.split(':').str[0].astype(int)

        # Calculate hourly averages
        hourly_avg = self.data.groupby('Hour')[self.pollutant_cols].mean()

        # Find AM peak (max before 12) and PM peak (max after or equal 12)
        am_peak_hour = hourly_avg.loc[hourly_avg.index < 12, 'Nitrogen Dioxide (ug/m3)'].idxmax()
        pm_peak_hour = hourly_avg.loc[hourly_avg.index >= 12, 'Nitrogen Dioxide (ug/m3)'].idxmax()

        # Create line plot showing hourly trends
        plt.figure(figsize=(14, 7))
        for col in self.pollutant_cols:  # Use pollutant_cols instead of hourly_avg.columns
            plt.plot(hourly_avg.index, hourly_avg[col], label=col)

        # Set axis limits
        plt.xlim(hourly_avg.index.min(), hourly_avg.index.max())

        # Highlight detected AM and PM peaks
        plt.axvspan(am_peak_hour-1, am_peak_hour+1, color='red', alpha=0.1,
                label=f'AM Rush ({am_peak_hour-1}-{am_peak_hour+1})')
        plt.axvspan(pm_peak_hour-2, pm_peak_hour, color='orange', alpha=0.1,
                label=f'PM Rush ({pm_peak_hour-2}-{pm_peak_hour})')

        # Add clear labels and formatting
        plt.title('Hourly Pollution Trends: Traffic Rush Hour Impact')
        plt.xlabel('Hour of Day (GMT)')
        plt.ylabel('Concentration (µg/m³)')
        plt.xticks(range(0, 23))  # Show all 23 hours clearly
        plt.grid(True)
        plt.legend(title='Pollutants', labels=self.pollutant_cols, bbox_to_anchor=(1.05, 1), loc='upper left')

        # Save visualisation if filename provided
        if save_as:
            plt.savefig(f"{save_as}_hourly_trends.{img_format}", bbox_inches='tight')
        plt.show()
        return hourly_avg

# INSIGHT 3 - Dominant pollutants with annotations
    def dominant_pollutant_analysis(self, save_as=None, img_format='png'):
        """
        Identify which pollutants most frequently dominate air quality concerns.

        Determines the dominant pollutant (highest concentration) for each month
        and visualises their frequency distribution with percentage annotations
        to highlight the most persistent pollution sources.

        Parameters:
            save_as (str, optional): Base filename for saving the visualisation
            img_format (str, optional): Image format for saved plot ('png', 'pdf', etc.)

        Returns:
            Series: Count of months where each pollutant was dominant
        """
        # Calculate monthly means
        monthly_means = self.data.groupby(pd.Grouper(key='Month', freq='ME'))[self.pollutant_cols].mean()

        # Change approach to exclude columns from consideration
        exclude_pollutants = ['Oxides of Nitrogen (ug/m3)', 'Nitrogen Dioxide (ug/m3)']
        filtered_cols = [col for col in self.pollutant_cols if col not in exclude_pollutants]

        # Find dominant pollutant among filtered columns
        monthly_means['Dominant Pollutant'] = monthly_means[filtered_cols].idxmax(axis=1)
        counts = monthly_means['Dominant Pollutant'].value_counts(normalize=True) * 100

        # Create bar plot showing distribution of dominant pollutants
        plt.figure(figsize=(10, 6))
        ax = sns.barplot(
            x=counts.values,
            y=counts.index,
            hue=counts.index,  # Assign y variable to hue
            palette='viridis',
            legend=False  # Remove redundant legend
        )

        # Annotate percentages
        for p in ax.patches:
            ax.annotate(f'{p.get_width():.1f}%', (p.get_width() + 1, p.get_y() + 0.5))

        # Add clear labels
        plt.title('Dominant Pollutants (Excluding NOx and NO₂)')
        plt.xlabel('Percentage of Months')

        # Save visualisation if filename provided
        if save_as:
            plt.savefig(f"{save_as}_dominant_pollutants.{img_format}", bbox_inches='tight')
        plt.show()
        return counts



### STEP 4: DATA VISUALISATION
if __name__ == "__main__":
    # Initialise air quality cleaner with dataset location
    cleaner = AirQualityCleaner('air-quality-london_dataset.xlsx', 'Time of day per month', skiprows=1)

    # Execute data cleaning pipeline
    cleaner.clean_dates()
    cleaner.clean()

    # Generate insights with professional visualisations
    cleaner.monthly_pollution_trends(save_as='london_air', img_format='pdf')
    cleaner.analyse_hourly_trends(save_as='london_air', img_format='pdf')
    cleaner.dominant_pollutant_analysis(save_as='london_air', img_format='pdf')

    # Save cleaned dataset for further analysis
    cleaner.save('cleaned_air_quality.csv')


FileNotFoundError: [Errno 2] No such file or directory: 'air-quality-london_dataset.xlsx'